# 04d · Baseline only sanity check (C5)

**Purpose.** Fit the same heads on the baseline part only (before any gas or scent is presented). A fair task scores near chance: 10% for GasWindTunnel multiclass, 25% for EverydayScents multiclass.

**Inputs**
- GasWindTunnel: pre-release segment of each recording, (600, 72, T_base)
- EverydayScents: baseline phase, (144, 11, 60)

**Outputs**
- `results/04d_baseline_sanity_check.csv`

## Colab setup
Run this first on Google Colab (Runtime → Change runtime type → **GPU**). Before running, upload
**`results.zip`** (results of notebooks 01-03) to `/content` with the Files panel on the left. The cell clones the
repo, downloads the processed data from the shared Drive folder, unpacks the results, and moves into this
notebook's folder so the rest runs unchanged. On a laptop it does nothing.

In [1]:
import glob
import os
import shutil
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
REPO = "/content/gas-repr-guide"
REPO_URL = "https://github.com/YasserAlbogami/gas-repr-guide.git"
DATA_URL = "https://drive.google.com/drive/folders/1rNIcZkHDFhH2dMVq8TgNWt_0IH2yw2_I"   # processed data (Yasser)
DATA_DIRS = ["gas_wind_tunnel", "everyday_scents"]

if IN_COLAB:
    # 1. code (src/). Tried in this order:
    #    a) clone the GitHub repo (works when the repo is public)
    #    b) clone with a GitHub token saved in Colab Secrets (key icon on the left) as GITHUB_TOKEN
    #    c) src.zip uploaded to /content (Files panel on the left)
    if not os.path.exists(f"{REPO}/src"):
        env = dict(os.environ, GIT_TERMINAL_PROMPT="0")           # fail instead of asking for a password
        ok = subprocess.run(["git", "clone", "-q", REPO_URL, REPO], env=env).returncode == 0
        if not ok:
            token = None
            try:
                from google.colab import userdata
                token = userdata.get("GITHUB_TOKEN")
            except Exception:
                pass
            if token:
                url = REPO_URL.replace("https://", f"https://{token}@")
                ok = subprocess.run(["git", "clone", "-q", url, REPO], env=env).returncode == 0
        if not ok and os.path.exists("/content/src.zip"):
            os.makedirs(f"{REPO}/notebooks/03_encoder", exist_ok=True)
            shutil.unpack_archive("/content/src.zip", REPO)
            ok = True
        if not ok:
            raise RuntimeError("Could not get the code: the repo is private. Make it public, or add GITHUB_TOKEN "
                               "in Colab Secrets, or upload src.zip to /content, then run this cell again.")
    elif os.path.exists(f"{REPO}/.git"):
        subprocess.run(["git", "-C", REPO, "pull", "-q"], check=False)
    os.makedirs(f"{REPO}/results", exist_ok=True)

    # 2. processed data (only downloaded once per Colab session)
    processed = f"{REPO}/data/processed"
    if not all(os.path.exists(f"{processed}/{d}/X_signals.npy") for d in DATA_DIRS):
        import gdown
        tmp = "/content/processed_download"
        gdown.download_folder(DATA_URL, output=tmp, quiet=False)
        for d in DATA_DIRS:
            found = glob.glob(f"{tmp}/**/{d}", recursive=True)
            assert found, f"folder {d} not found in the Drive download"
            shutil.copytree(found[0], f"{processed}/{d}", dirs_exist_ok=True)
    for d in DATA_DIRS:
        files = sorted(os.listdir(f"{processed}/{d}"))
        print(d, files)
        for f in ["X_signals.npy", "meta.csv", "info.json"]:
            assert f in files, f"{d}/{f} is missing"

    # 2b. results of the earlier notebooks (01, 02, 03 and the 04 ones already run), from results.zip
    #     uploaded to /content (Files panel on the left). The zip holds a results/ folder (+ report/tables/).
    if os.path.exists("/content/results.zip"):
        shutil.unpack_archive("/content/results.zip", REPO)
    have = sorted(os.path.basename(f) for f in glob.glob(f"{REPO}/results/*.csv"))
    print("results:", have)
    main = ["01a_gwt_raw", "01b_es_raw", "02b_gwt_handcrafted", "02c_es_handcrafted",
            "03a_gwt_encoder", "03b_es_encoder", "03c_end_to_end_reference"]
    missing = [n for n in main if f"{n}.csv" not in have]
    assert not missing, f"upload results.zip to /content first (missing: {missing})"

    # 3. run the rest of the notebook from its own folder, as on a laptop (so ../.. = project root)
    os.chdir(f"{REPO}/notebooks/04_comparisons")

    import torch
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
          else "NONE, switch on Runtime > Change runtime type > GPU")

In [2]:
import sys
sys.path.append("../..")          # project root, so that `import src` works

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path("../..")
from src.cv import TASKS, load_baseline_level, load_processed, precomputed, run_cv, save_results, summarize
from src.features.raw import raw_features

NAME = "04d_baseline_sanity_check"

## C5: train the same heads on the baseline part only
The baseline is recorded **before** any gas or scent is presented, so a fair task should score near chance
(macro F1 about 1/K: 0.10 for GasWindTunnel multiclass, 0.25 for EverydayScents multiclass).
The baseline is used **without** baseline correction (the resting level is added back), because the resting level
is exactly what could leak the label (recording batches over months, black pepper recorded on 4 days only).

Two baseline-only representations:
- `B1_baseline_raw`: the uncorrected baseline curves, downsampled (R1 on the baseline)
- `B2_baseline_level`: the resting level only, one number per sensor (R2 feature 1)

In [3]:
parts = []
for dataset in ["gwt", "es"]:
    X, meta, info = load_processed(dataset)                       # (N, C, T)
    levels = load_baseline_level(dataset)                         # (N, C)
    n_base = info["n_base"]
    X_base = X[:, :, :n_base] + levels[:, :, None]                # (N, C, n_base) uncorrected baseline
    F_b1 = raw_features(X_base, max(2, n_base // 3))              # (N, C * n_base/3)
    F_b2 = levels                                                 # (N, C)
    for task in TASKS:
        for rep, F in [("B1_baseline_raw", F_b1), ("B2_baseline_level", F_b2)]:
            s, _ = run_cv(F, meta, dataset, task, rep, precomputed, verbose=False)
            parts.append(s)
scores = pd.concat(parts, ignore_index=True)
save_results(scores, None, NAME)

/Users/saeedal-asiri/Desktop/data analytics/gas-repr-guide/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


/Users/saeedal-asiri/Desktop/data analytics/gas-repr-guide/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/saeedal-asiri/Desktop/data analytics/gas-repr-guide/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


/Users/saeedal-asiri/Desktop/data analytics/gas-repr-guide/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


/Users/saeedal-asiri/Desktop/data analytics/gas-repr-guide/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


/Users/saeedal-asiri/Desktop/data analytics/gas-repr-guide/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


/Users/saeedal-asiri/Desktop/data analytics/gas-repr-guide/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/saeedal-asiri/Desktop/data analytics/gas-repr-guide/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/saeedal-asiri/Desktop/data analytics/gas-repr-guide/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/saeedal-asiri/Desktop/data analytics/gas-repr-guide/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: 

... (32 outputs left out)

/Users/saeedal-asiri/Desktop/data analytics/gas-repr-guide/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/saeedal-asiri/Desktop/data analytics/gas-repr-guide/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


/Users/saeedal-asiri/Desktop/data analytics/gas-repr-guide/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/saeedal-asiri/Desktop/data analytics/gas-repr-guide/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


/Users/saeedal-asiri/Desktop/data analytics/gas-repr-guide/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


/Users/saeedal-asiri/Desktop/data analytics/gas-repr-guide/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


In [4]:
chance = {("gwt", "multi"): 0.10, ("es", "multi"): 0.25, ("gwt", "binary"): 0.5, ("es", "binary"): 0.5}
table = summarize(scores, "balanced_acc")
table["chance (balanced acc)"] = [chance[(d, t)] for d, t, _ in table.index]
print("balanced accuracy (chance level is exact for this metric)")
display(table)
print("macro F1")
display(summarize(scores))

balanced accuracy (chance level is exact for this metric)


head                                     logreg            mlp             rf  \
dataset task   rep                                                              
es      binary B1_baseline_raw    0.573 ± 0.131  0.591 ± 0.089  0.562 ± 0.100   
               B2_baseline_level  0.588 ± 0.134  0.561 ± 0.084  0.569 ± 0.103   
        multi  B1_baseline_raw    0.362 ± 0.133  0.281 ± 0.118  0.283 ± 0.126   
               B2_baseline_level  0.345 ± 0.091  0.271 ± 0.121  0.289 ± 0.119   
gwt     binary B1_baseline_raw    0.999 ± 0.002  0.997 ± 0.011  0.991 ± 0.017   
               B2_baseline_level  1.000 ± 0.001  1.000 ± 0.000  0.992 ± 0.017   
        multi  B1_baseline_raw    0.992 ± 0.007  0.989 ± 0.009  0.989 ± 0.009   
               B2_baseline_level  0.993 ± 0.006  0.992 ± 0.007  0.990 ± 0.010   

head                                        svm  chance (balanced acc)  
dataset task   rep                                                      
es      binary B1_baseline_raw    0.614 ± 0.103                   0.50  
               B2_baseline_level  0.648 ± 0.108                   0.50  
        multi  B1_baseline_raw    0.299 ± 0.105                   0.25  
               B2_baseline_level  0.322 ± 0.112                   0.25  
gwt     binary B1_baseline_raw    1.000 ± 0.000                   0.50  
               B2_baseline_level  1.000 ± 0.000                   0.50  
        multi  B1_baseline_raw    0.994 ± 0.010                   0.10  
               B2_baseline_level  0.994 ± 0.010                   0.10

macro F1


head                                     logreg            mlp             rf  \
dataset task   rep                                                              
es      binary B1_baseline_raw    0.550 ± 0.135  0.573 ± 0.101  0.539 ± 0.112   
               B2_baseline_level  0.558 ± 0.136  0.537 ± 0.089  0.547 ± 0.112   
        multi  B1_baseline_raw    0.336 ± 0.125  0.263 ± 0.113  0.263 ± 0.122   
               B2_baseline_level  0.315 ± 0.093  0.259 ± 0.120  0.271 ± 0.110   
gwt     binary B1_baseline_raw    0.997 ± 0.008  0.997 ± 0.008  0.994 ± 0.011   
               B2_baseline_level  0.999 ± 0.006  1.000 ± 0.000  0.995 ± 0.010   
        multi  B1_baseline_raw    0.992 ± 0.007  0.989 ± 0.009  0.989 ± 0.009   
               B2_baseline_level  0.993 ± 0.006  0.992 ± 0.007  0.990 ± 0.010   

head                                        svm  
dataset task   rep                               
es      binary B1_baseline_raw    0.599 ± 0.117  
               B2_baseline_level  0.629 ± 0.120  
        multi  B1_baseline_raw    0.280 ± 0.097  
               B2_baseline_level  0.303 ± 0.111  
gwt     binary B1_baseline_raw    1.000 ± 0.000  
               B2_baseline_level  1.000 ± 0.000  
        multi  B1_baseline_raw    0.994 ± 0.010  
               B2_baseline_level  0.994 ± 0.010

In [5]:
best = scores.groupby(["dataset", "task", "rep", "head"])["balanced_acc"].mean().groupby(["dataset", "task"]).max()
for (d, t), v in best.items():
    flag = "ABOVE CHANCE: possible leak, report it" if v > chance[(d, t)] + 0.10 else "near chance: ok"
    print(f"{d} {t}: best baseline-only balanced accuracy {v:.3f} (chance {chance[(d, t)]:.2f}) -> {flag}")

es binary: best baseline-only balanced accuracy 0.648 (chance 0.50) -> ABOVE CHANCE: possible leak, report it
es multi: best baseline-only balanced accuracy 0.362 (chance 0.25) -> ABOVE CHANCE: possible leak, report it
gwt binary: best baseline-only balanced accuracy 1.000 (chance 0.50) -> ABOVE CHANCE: possible leak, report it
gwt multi: best baseline-only balanced accuracy 0.994 (chance 0.10) -> ABOVE CHANCE: possible leak, report it


## Colab: save the outputs
Colab deletes everything when the session ends. This zips `results/` and `report/` into `outputs_<notebook>.zip`,
copies it to your Google Drive (`MyDrive/gas-repr-guide/`) and downloads it. On a laptop it does nothing.

In [6]:
if IN_COLAB:
    out_zip = shutil.make_archive(f"/content/outputs_04d_baseline_sanity_check", "zip", REPO, "results")
    shutil.make_archive(f"/content/report_04d_baseline_sanity_check", "zip", REPO, "report")
    files_out = [out_zip, f"/content/report_04d_baseline_sanity_check.zip"]
    print(files_out)
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        out_dir = "/content/drive/MyDrive/gas-repr-guide"
        os.makedirs(out_dir, exist_ok=True)
        for f in files_out:
            shutil.copy(f, out_dir)
        print("copied to", out_dir)
    except Exception as e:
        print("Drive copy failed:", e)
    from google.colab import files
    for f in files_out:
        files.download(f)